# perchv2-pytorch quickstart (ONNX-converted backbone)

This is the **recommended** way to use this repo. It walks through all three usage modes, plus **partial fine-tuning** (a capability the native `timm`-based backbone doesn't have at the same granularity), using the ONNX-converted backbone (`PerchONNXBackbone`/`PerchONNXClassifier`/`PerchONNXEmbedder`) -- confirmed 1.00000000 cosine similarity against Google's official released model, and fully differentiable.

1. **Frozen features** -- `PerchONNXEmbedder`, no training at all.
2. **Linear probing** -- backbone frozen, only a new head is trained.
3. **Full fine-tuning** -- backbone unfrozen, gradients flow through everything.
4. **Partial fine-tuning** -- freeze specific early blocks, train the rest.

Everything below runs against a synthetic 3-class sine-tone dataset with no download and no real audio required, so you can run this notebook top to bottom with nothing but `pip install -e ".[onnx]"` and a `perch_v2.onnx` file (see the README's "Getting the weights" section).

**This notebook needs the `perch_v2.onnx` file to run** -- the whole model, including its frontend, is converted from that one file. Get it from [`justinchuby/Perch-onnx`](https://huggingface.co/justinchuby/Perch-onnx/tree/main), and place it at `weights/perch_v2.onnx`.


In [3]:
import sys
import warnings
from pathlib import Path

import torch
import torch.nn as nn
from torch.utils.data import DataLoader

# examples/toy_dataset.py isn't part of the installed package -- it's a
# standalone example module, so we add examples/ to the path to import it.
REPO_ROOT = Path.cwd().parent if (Path.cwd() / "perchv2_pytorch").exists() is False else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))
sys.path.insert(0, str(REPO_ROOT / "examples"))

from perchv2_pytorch import PerchONNXEmbedder, PerchONNXBackbone, PerchONNXClassifier
from toy_dataset import ToySineDataset

warnings.filterwarnings("ignore", message="CUDA initialization.*", category=UserWarning)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# Point this at your own perch_v2.onnx -- see the README's "Getting the
# weights" section for where to get it (this repo doesn't redistribute it).
ONNX_PATH = REPO_ROOT / "weights" / "perch_v2.onnx"
CACHE_DIR = REPO_ROOT / "weights" / "onnx_cache"

if not ONNX_PATH.exists():
    print(f"WARNING: {ONNX_PATH} not found -- every cell below will fail until")
    print("you place a real perch_v2.onnx there.")


Using device: cpu


## Section 1 -- Frozen features

`PerchONNXEmbedder` wraps the full converted graph (frontend + backbone, no head) with a simple `embed`-style interface. This is the mode to use if you just want fixed embeddings for your own downstream classifier, clustering, or search index -- no training happens here.

Unlike the native backbone's `PerchFrontend` + `Perch2Backbone` split, this takes raw waveform directly -- framing, windowing, and the mel filterbank are all part of the converted graph.

**Note on speed:** the cell below will take several seconds the first time it runs -- converting the raw ONNX graph has real, one-time overhead. Every model built later in this notebook passes the same `cache_dir`, so they reuse that conversion instead of repeating it and should be much faster.


In [4]:
embedder = PerchONNXEmbedder(str(ONNX_PATH), cache_dir=str(CACHE_DIR)).to(device)
embedder.eval()

# Perch v2 expects 5s mono clips at 32kHz = 160,000 samples.
batch = torch.zeros(4, 160_000).to(device)

with torch.no_grad():
    embeddings = embedder(batch)

print("Embedding shape:", embeddings.shape)  # (4, 1536)


Embedding shape: torch.Size([4, 1536])


## The toy dataset used below

`ToySineDataset` (in `examples/toy_dataset.py`) generates 3 classes of synthetic audio -- each class is a distinct sine-tone frequency (500Hz / 1500Hz / 4000Hz) plus a little Gaussian noise, correctly shaped as 5s/32kHz clips. It exists purely to exercise the training loop mechanics without needing real recordings.

**This is not biologically meaningful data.** Don't read anything into the accuracy numbers below.


In [5]:
# Swap this out for your own data: any torch.utils.data.Dataset that
# returns (waveform, label) pairs, where waveform is a 1D float32 tensor
# of shape (160000,) -- 5 seconds of mono audio at 32kHz -- and label is
# an integer class index.
dataset = ToySineDataset(n_per_class=20)  # placeholder synthetic data, see examples/toy_dataset.py
train_loader = DataLoader(dataset, batch_size=8, shuffle=True)

print(f"{len(dataset)} clips, {dataset.num_classes} classes")


60 clips, 3 classes


## Section 2 -- Linear probing

`mode="linear_probe"` freezes every backbone parameter and trains only the new head. Fast, low-data-friendly, and a good baseline before paying for a full fine-tune.


In [5]:
linear_probe_model = PerchONNXClassifier(
    num_classes=dataset.num_classes,
    onnx_path=str(ONNX_PATH),
    mode="linear_probe",
    cache_dir=str(CACHE_DIR),
).to(device)

trainable = [n for n, p in linear_probe_model.named_parameters() if p.requires_grad]
print(f"{len(trainable)} trainable parameter tensors (should just be the head's)")

optimizer = torch.optim.Adam(
    filter(lambda p: p.requires_grad, linear_probe_model.parameters()), lr=1e-3
)
criterion = nn.CrossEntropyLoss()

EPOCHS = 5
linear_probe_model.train()
for epoch in range(EPOCHS):
    total_loss, correct, total = 0.0, 0, 0
    for waveforms, labels in train_loader:
        waveforms, labels = waveforms.to(device), labels.to(device)

        optimizer.zero_grad()
        logits = linear_probe_model(waveforms)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * waveforms.size(0)
        correct += (logits.argmax(dim=1) == labels).sum().item()
        total += waveforms.size(0)

    print(f"epoch {epoch+1}/{EPOCHS}  loss={total_loss/total:.4f}  acc={correct/total:.2%}")


2 trainable parameter tensors (should just be the head's)
epoch 1/5  loss=0.8773  acc=93.33%
epoch 2/5  loss=0.5013  acc=100.00%
epoch 3/5  loss=0.2879  acc=100.00%
epoch 4/5  loss=0.1792  acc=100.00%
epoch 5/5  loss=0.1197  acc=100.00%


## Section 3 -- Full fine-tuning

`mode="finetune"` unfreezes the entire backbone -- gradients flow through everything, not just the head. Each epoch prints `backbone_grad_norm` as a direct check that this is actually happening (it would be absent in `frozen`/`linear_probe` mode).


In [6]:
finetune_model = PerchONNXClassifier(
    num_classes=dataset.num_classes,
    onnx_path=str(ONNX_PATH),
    mode="finetune",
    cache_dir=str(CACHE_DIR),
).to(device)

optimizer = torch.optim.AdamW(
    [
        {"params": finetune_model.backbone.parameters(), "lr": 1e-5},
        {"params": finetune_model.head.parameters(), "lr": 1e-3},
    ]
)
criterion = nn.CrossEntropyLoss()

EPOCHS = 5
finetune_model.train()
for epoch in range(EPOCHS):
    total_loss, correct, total = 0.0, 0, 0
    for waveforms, labels in train_loader:
        waveforms, labels = waveforms.to(device), labels.to(device)

        optimizer.zero_grad()
        logits = finetune_model(waveforms)
        loss = criterion(logits, labels)
        loss.backward()

        backbone_grad_norm = sum(
            p.grad.norm().item()
            for p in finetune_model.backbone.parameters()
            if p.grad is not None
        )

        optimizer.step()

        total_loss += loss.item() * waveforms.size(0)
        correct += (logits.argmax(dim=1) == labels).sum().item()
        total += waveforms.size(0)

    print(
        f"epoch {epoch+1}/{EPOCHS}  loss={total_loss/total:.4f}  "
        f"acc={correct/total:.2%}  backbone_grad_norm={backbone_grad_norm:.4f}"
    )


epoch 1/5  loss=0.9319  acc=80.00%  backbone_grad_norm=7.8281
epoch 2/5  loss=0.5284  acc=100.00%  backbone_grad_norm=8.8775
epoch 3/5  loss=0.2783  acc=100.00%  backbone_grad_norm=10.3451
epoch 4/5  loss=0.1459  acc=100.00%  backbone_grad_norm=4.6917
epoch 5/5  loss=0.0820  acc=100.00%  backbone_grad_norm=4.4000


## Section 4 -- Partial fine-tuning (freeze specific blocks)

This is the capability the ONNX-converted backbone has that the native `timm` backbone doesn't support at this granularity: freeze the stem and early blocks (which likely encode more generic, transferable features) while training later blocks and the head (which likely need more domain-specific adaptation).

`freeze_up_to_block(n)` freezes the stem and blocks `0` through `n-1`, leaving blocks `n` through `25` and the head trainable. Under the hood this works by reconstructing block-level structure from the ONNX graph's own tensor-naming metadata -- see `build_block_map()` in `perchv2_pytorch/onnx_backbone.py` if you want the details.


In [7]:
partial_backbone = PerchONNXBackbone(str(ONNX_PATH), cache_dir=str(CACHE_DIR))

# Freeze the stem + blocks 0-12 (roughly the first half of the network),
# train blocks 13-25 and everything after.
frozen, trainable = partial_backbone.freeze_up_to_block(13)
print(f"freeze_up_to_block(13): {frozen} frozen, {trainable} trainable parameters")

# Try a different split
frozen, trainable = partial_backbone.freeze_up_to_block(20)
print(f"freeze_up_to_block(20): {frozen} frozen, {trainable} trainable parameters")

# Build a classifier on top and train it -- identical loop to Section 3,
# just with a partially-frozen backbone instead of fully unfrozen.
partial_head = nn.Linear(1536, dataset.num_classes).to(device)
partial_backbone = partial_backbone.to(device)

optimizer = torch.optim.AdamW(
    [
        {"params": filter(lambda p: p.requires_grad, partial_backbone.parameters()), "lr": 1e-5},
        {"params": partial_head.parameters(), "lr": 1e-3},
    ]
)
criterion = nn.CrossEntropyLoss()

EPOCHS = 5
partial_backbone.train()
for epoch in range(EPOCHS):
    total_loss, correct, total = 0.0, 0, 0
    for waveforms, labels in train_loader:
        waveforms, labels = waveforms.to(device), labels.to(device)

        optimizer.zero_grad()
        emb = partial_backbone(waveforms)
        logits = partial_head(emb)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * waveforms.size(0)
        correct += (logits.argmax(dim=1) == labels).sum().item()
        total += waveforms.size(0)

    print(f"epoch {epoch+1}/{EPOCHS}  loss={total_loss/total:.4f}  acc={correct/total:.2%}")


freeze_up_to_block(13): 77 frozen, 80 trainable parameters
freeze_up_to_block(20): 119 frozen, 38 trainable parameters
epoch 1/5  loss=0.9182  acc=78.33%
epoch 2/5  loss=0.5234  acc=100.00%
epoch 3/5  loss=0.2932  acc=100.00%
epoch 4/5  loss=0.1717  acc=100.00%
epoch 5/5  loss=0.1069  acc=100.00%


## Using your own data and weights

**1. Get a real Perch v2 `.onnx` file** -- see the README's "Getting the weights" section. Place it wherever `ONNX_PATH` points, or edit that variable directly.

**2. Replace `ToySineDataset` with your own `Dataset`.** It needs to return `(waveform, label)` pairs where:
   - `waveform` is a 1D `float32` tensor of shape `(160000,)` -- 5 seconds of mono audio at 32kHz. Resample and pad/trim if your audio differs.
   - `label` is an integer class index.

   A minimal real-audio version:
   ```python
   import torchaudio

   class MyAudioDataset(torch.utils.data.Dataset):
       def __init__(self, filepaths, labels):
           self.filepaths = filepaths
           self.labels = labels

       def __len__(self):
           return len(self.filepaths)

       def __getitem__(self, idx):
           waveform, sr = torchaudio.load(self.filepaths[idx])
           waveform = waveform.mean(dim=0)  # mono
           if sr != 32000:
               waveform = torchaudio.functional.resample(waveform, sr, 32000)
           target_len = 160_000
           if waveform.shape[0] < target_len:
               waveform = torch.nn.functional.pad(waveform, (0, target_len - waveform.shape[0]))
           else:
               waveform = waveform[:target_len]
           return waveform, self.labels[idx]
   ```

**3. Update `num_classes`.** Comes from `dataset.num_classes` automatically in the cells above -- expose the same attribute on your own `Dataset`, or pass an integer directly.

**4. Pick a mode.** Frozen (`PerchONNXEmbedder`), linear probe or full fine-tune (`PerchONNXClassifier(mode=...)`), or partial fine-tuning (`PerchONNXBackbone.freeze_up_to_block(n)`, Section 4 above) -- everything else stays the same.
